In [1]:
%load_ext autoreload
%autoreload 2

import re
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import spearmanr

import torch
import sys

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
from scripts.dataset_class import SimpleDataset
from scripts.preprocessing import Preprocessor

import nested_cv_metrics_opt as n_cv_opt

In [4]:
oof_preds, fold_meta, meta_df, info = n_cv_opt.load_artifact("results/plain_hibou_l_spec.pkl")

fm_df = pd.DataFrame(fold_meta).sort_values(by='metric')
fm_df = n_cv_opt.add_preproc_key(fm_df)
fm_df

Loaded artifact from results/plain_hibou_l_spec.pkl
  run_info: {'timestamp': '2026-10-03 14:09:23', 'outer_seed': 155, 'identities': ['hibou_l_spec'], 'configs': "normalization:[False, True] | channel_mode:['gray', 'hed'] | flip_intensity:[False, True] | match_histogram:[False, True] | clahe:[False, True] | smoothing:[False, True]", 'n_outer_splits': 5, 'n_inner_splits': 4}


,preproc_cfg,fold,metric,config
0,0 | rgb | 0 | 0 | 0 | 0 | layer_20 | cos,0,hibou_l_spec,"{'normalization': False, 'channel_mode': 'rgb'..."
1,0 | rgb | 0 | 0 | 0 | 0 | layer_20 | cos,1,hibou_l_spec,"{'normalization': False, 'channel_mode': 'rgb'..."
2,0 | rgb | 0 | 0 | 0 | 0 | layer_20 | cos,2,hibou_l_spec,"{'normalization': False, 'channel_mode': 'rgb'..."
3,0 | rgb | 0 | 0 | 0 | 0 | layer_20 | cos,3,hibou_l_spec,"{'normalization': False, 'channel_mode': 'rgb'..."
4,0 | rgb | 0 | 0 | 0 | 0 | layer_20 | cos,4,hibou_l_spec,"{'normalization': False, 'channel_mode': 'rgb'..."


In [5]:
n_cv_opt.evaluate_oof(oof_preds, meta_df)

,metric,spearman_class3,auc_bad_vs_rest,auc_good_vs_rest,auc_3class_ovo,spearman_raw_1_5
0,hibou_l_spec,0.672799,0.88925,0.825851,0.814938,0.690061


## Failure Analysis

Сформировать несколько панелей: 

    1. representative Good/Borderline/Bad пары с expert score и scores HAPS/NCC/LPIPS/SSIM;
    2. cases where HAPS agrees with expert and baselines fail; 
    3. cases where HAPS fails;
    
Для каждой пары показывать H&E, HER2, возможно overlay/edge/tissue mask, expert class, metric ranks.

In [ ]:
csv = "/beegfs/home/f.gubanov/f.gubanov/bimai_lab/filtration_imgs/exp_full.csv"
ds = SimpleDataset(csv, dist_col="Similarity_Score", class_col="class3", return_meta=True)
all_pairs, meta_df = ncv._cache_dataset(ds)